In [2]:
# ============================================================
# CELL 1 — Setup, KEBS limits, county/source cleaning, load data
# ============================================================

import os
import re
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Project root (local path). Used for output folders + DB path.
PROJECT_DIR = r"C:\Users\user\Desktop\Final Project"
os.chdir(PROJECT_DIR)
os.makedirs("output/charts", exist_ok=True)   # PNG charts
os.makedirs("output/results", exist_ok=True)  # CSV/TXT results

# Global chart style — applied to every chart in this file
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (11, 6)
plt.rcParams["font.size"] = 11
plt.rcParams["axes.titleweight"] = "bold"

# KEBS limits — single source of truth used by every chart
KEBS = {
    "ph_min": 6.5, "ph_max": 8.5,
    "conductivity": 2500,
    "tds": 1000,
    "alkalinity": 500,
    "colour": 15,
}

# All 47 official Kenyan counties
KENYA_COUNTIES = [
    "Baringo", "Bomet", "Bungoma", "Busia", "Elgeyo-Marakwet", "Embu",
    "Garissa", "Homa Bay", "Isiolo", "Kajiado", "Kakamega", "Kericho",
    "Kiambu", "Kilifi", "Kirinyaga", "Kisii", "Kisumu", "Kitui",
    "Kwale", "Laikipia", "Lamu", "Machakos", "Makueni", "Mandera",
    "Marsabit", "Meru", "Migori", "Mombasa", "Murang'a", "Nairobi",
    "Nakuru", "Nandi", "Narok", "Nyamira", "Nyandarua", "Nyeri",
    "Samburu", "Siaya", "Taita-Taveta", "Tana River", "Tharaka-Nithi",
    "Trans Nzoia", "Turkana", "Uasin Gishu", "Vihiga", "Wajir", "West Pokot"
]

# Lowercase, remove punctuation/spaces — makes matching robust
def _norm(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower())

_KENYA_LOOKUP = {_norm(c): c for c in KENYA_COUNTIES}

# Known aliases (towns/misspellings) -> official county
_VARIANT_MAP = {
    "homabay": "Homa Bay", "sondu": "Kisumu", "nyando": "Kisumu",
    "kisumueast": "Kisumu", "lodwar": "Turkana", "lokichoggio": "Turkana",
    "mwingi": "Kitui", "thika": "Kiambu", "taitataveta": "Taita-Taveta",
    "tharakantihi": "Tharaka-Nithi", "elgeyomarakwet": "Elgeyo-Marakwet",
    "muranga": "Murang'a",
}

# Values to DROP — non-Kenyan locations and old provinces
_NON_KENYAN = {
    "somalia", "mogadishu", "mogadishusomalia",
    "moshitanzania", "iringatanzania", "iringa",
    "northernbaherelghazalsouthernsudan", "central", "riftvalley",
}

# Map raw county -> canonical name, or None to drop
def map_county(raw):
    if pd.isna(raw):
        return None
    key = _norm(raw)
    if key in _KENYA_LOOKUP:  return _KENYA_LOOKUP[key]
    if key in _VARIANT_MAP:   return _VARIANT_MAP[key]
    if key in _NON_KENYAN:    return None
    return str(raw).strip()

# Collapse messy source labels into 4 buckets
def clean_source(s):
    s = str(s).lower().strip()
    if "bore" in s or "b/hole" in s:                                   return "Borehole"
    if "shallow" in s or "well" in s:                                  return "Shallow Well"
    if any(k in s for k in ("pipe", "tap", "treated", "domestic")):    return "Piped/Treated"
    if any(k in s for k in ("effluent", "waste", "sewage", "influent")): return "Effluent"
    return None

# ---- Load raw data from SQLite ----
DB_PATH = os.path.join(PROJECT_DIR, "data", "processed", "kenya_water.db")
conn = sqlite3.connect(DB_PATH)
kewi_raw = pd.read_sql_query("SELECT * FROM kewi_water_tests", conn)
conn.close()

# Normalize column names: strip, lowercase, spaces -> underscores
kewi_raw.columns = [c.strip().lower().replace(" ", "_") for c in kewi_raw.columns]

# Coerce the 5 parameters to numeric (blanks -> NaN)
for _c in ["ph", "conductivity?s/cm", "total_dissolved_solids", "alkalinity", "colour"]:
    if _c in kewi_raw.columns:
        kewi_raw[_c + "_num"] = pd.to_numeric(kewi_raw[_c], errors="coerce")

if "county" in kewi_raw.columns:
    kewi_raw["county_clean"] = kewi_raw["county"].apply(map_county)

if "type_of_h2o" in kewi_raw.columns:
    kewi_raw["source_clean"] = kewi_raw["type_of_h2o"].apply(clean_source)

# Drop non-Kenyan / old-province rows
kewi = kewi_raw[kewi_raw["county_clean"].notna()].copy()

# Column constants
PH = "ph_num"
COND = "conductivity?s/cm_num"
TDS  = "total_dissolved_solids_num"
ALK  = "alkalinity_num"
COL  = "colour_num"
COUNTY = "county_clean"
SOURCE = "source_clean"

# Parameter inventory — every later section loops over this
PARAMS = [
    ("pH",           PH,   "",      KEBS["ph_min"],  KEBS["ph_max"]),
    ("Conductivity", COND, "µS/cm", None,            KEBS["conductivity"]),
    ("TDS",          TDS,  "mg/L",  None,            KEBS["tds"]),
    ("Alkalinity",   ALK,  "mg/L",  None,            KEBS["alkalinity"]),
    ("Colour",       COL,  "TCU",   None,            KEBS["colour"]),
]

# ---- Sanity output ----
print("=" * 60)
print(f"Loaded {len(kewi)} samples (non-Kenyan rows removed)")
print("=" * 60)

print("\n=== Parameter coverage ===")
for name, col, unit, lo, hi in PARAMS:
    if col in kewi.columns:
        n = kewi[col].notna().sum()
        print(f"  {name:14s} {n:4d} / {len(kewi)}  ({100*n/len(kewi):.1f}%)")
    else:
        print(f"  {name:14s} column not found in DB")

Loaded 469 samples (non-Kenyan rows removed)

=== Parameter coverage ===
  pH              411 / 469  (87.6%)
  Conductivity    386 / 469  (82.3%)
  TDS             393 / 469  (83.8%)
  Alkalinity     column not found in DB
  Colour          383 / 469  (81.7%)


In [ ]:
# ============================================================
# CELL 2 — Descriptive statistics vs KEBS
# No charts here — just a printed + saved table.
# ============================================================

stats_rows = []
for name, col, unit, lo, hi in PARAMS:
    if col not in kewi.columns:
        continue
    v = kewi[col].dropna()
    if len(v) == 0:
        continue
    # Violation rule differs for pH (two-sided) vs the rest (one-sided upper)
    if lo is not None:
        bad = ((v < lo) | (v > hi)).sum()
        rule = f"{lo}–{hi}"
    else:
        bad = (v > hi).sum()
        rule = f"≤ {hi}"
    stats_rows.append({
        "Parameter": name,
        "Unit": unit,
        "n": len(v),
        "Missing": len(kewi) - len(v),
        "Min": round(v.min(), 2),
        "Q1": round(v.quantile(0.25), 2),
        "Median": round(v.median(), 2),
        "Mean": round(v.mean(), 2),
        "Q3": round(v.quantile(0.75), 2),
        "Max": round(v.max(), 2),
        "KEBS Rule": rule,
        "Violations": int(bad),
        "Violation %": round(100 * bad / len(v), 1),
    })

stats_df = pd.DataFrame(stats_rows)
print("\n=== Descriptive statistics vs KEBS ===")
print(stats_df.to_string(index=False))
stats_df.to_csv("output/results/descriptive_stats.csv", index=False)
print("\n✅ Saved: output/results/descriptive_stats.csv")